# 5.3 Kappa Agreement

This notebook calculates inter-annotator agreement between R3 and R8 for the round 1 manual annotation set.

The agreement is reported for:
- friction_type
- severity
- sentiment

In [11]:
import os
import pandas as pd
from sklearn.metrics import cohen_kappa_score
print("Current working directory:", os.getcwd())

Current working directory: c:\Users\Sialiang\OneDrive - The University of Sydney (Students)\capstone\usyd-03-2025-cs20-1\notebooks


In [13]:
# Read annotation files
R3_PATH = "../data/annotations/r3_manual_annotations_round1.csv"
R8_PATH = "../data/annotations/r8_manual_annotations_round1.csv"

r3 = pd.read_csv(R3_PATH)
r8 = pd.read_csv(R8_PATH)

print("R3 shape:", r3.shape)
print("R8 shape:", r8.shape)

display(r3.head())
display(r8.head())

R3 shape: (14, 21)
R8 shape: (14, 13)


,window_id,project,video_id,video_filename,tester_name,task_title,task_instructions,start_time,end_time,duration,...,narration_type,at_context_present,friction_type,severity,sentiment,confidence,primary_evidence,secondary_friction_type,notes,annotator
0,ramazankawish_wa_w075,department-of-premier-and-cabinet-wa,ramazankawish_wa,project_1f7f2859-ac90-4696-b89f-1077aa6040be_r...,ramazankawish,Providing Feedback via the Website,**Please use fake data for this task** **Ob...,5344.430,5414.24,69.810,...,navigation,yes,F1 Navigation / Findability Issue,medium,negative,medium,"""I'm struggling to find the feedback page""",NaN,Clear findability issue for feedback/complaint...,R3
1,gameoverdan_suncorp_w040,suncorp-insurance,gameoverdan_suncorp,project_73052603-e4d8-4aed-a9ce-cc5db15e6355_g...,gameoverdan,Equal Access to Critical Information – Emerge...,Assume you are an existing customer who is ant...,2719.229,2782.55,63.321,...,navigation,no,F1 Navigation / Findability Issue,medium,negative,medium,"""no, we're doing it online. Doing it online. C...",NaN,Repeated navigation attempts suggest difficult...,R3
2,Sharelinsonny_uq_w026,the-university-of-queensland,Sharelinsonny_uq,project_8aeb4cdb-106f-47c1-b6e6-867344ce673b_S...,Sharelinsonny,Commence Enrolment - Register an Account,### Objective This task evaluates how easy ...,1818.130,1880.76,62.630,...,feedback_evaluation,no,F2 Content Clarity Issue,medium,negative,medium,"""formal legal language, it's not very user-fri...",NaN,Clear content readability issue caused by lega...,R3
3,oliviamitchell22_suncorp_w007,suncorp-insurance,oliviamitchell22_suncorp,project_73052603-e4d8-4aed-a9ce-cc5db15e6355_o...,oliviamitchell22,Navigating AAMI’s Insurance Service Offering,## **Objective** We don't actually want you...,471.429,533.50,62.071,...,thinking_aloud,no,F2 Content Clarity Issue,medium,mixed,medium,"""What does that mean? ... I have no clue""",NaN,Insurance terminology clarity issue; positive ...,R3
4,giuliaclemente26_uq_w050,the-university-of-queensland,giuliaclemente26_uq,project_8aeb4cdb-106f-47c1-b6e6-867344ce673b_g...,giuliaclemente26,Managing Profile and Security Settings,### Objective This task evaluates how easy ...,3536.939,3597.30,60.361,...,feedback_evaluation,no,F3 Interaction / Control Issue,medium,negative,medium,"""I need to install something which is already ...",NaN,MFA setup creates workflow burden and potentia...,R3


,window_id,project,video_id,task_title,text,narration_type,at_context_present,friction_type,severity,sentiment,confidence,notes,annotator
0,ramazankawish_wa_w075,department-of-premier-and-cabinet-wa,ramazankawish_wa,Providing Feedback via the Website,For general complaints and feedback about the ...,thinking_aloud,no,F1,medium,negative,medium,Tester says they are struggling to find the fe...,R8
1,gameoverdan_suncorp_w040,suncorp-insurance,gameoverdan_suncorp,Equal Access to Critical Information – Emerge...,The homepage. I will say the homepage has done...,navigation,no,F1,low,mixed,medium,Tester navigates through multiple options and ...,R8
2,Sharelinsonny_uq_w026,the-university-of-queensland,Sharelinsonny_uq,Commence Enrolment - Register an Account,"So we can improve it based on your needs, pers...",feedback_evaluation,no,F2,medium,negative,high,Tester says the section is written in formal l...,R8
3,oliviamitchell22_suncorp_w007,suncorp-insurance,oliviamitchell22_suncorp,Navigating AAMI’s Insurance Service Offering,"Uh, what else is there? Third party car? What ...",feedback_evaluation,no,F2,low,mixed,medium,"Tester is confused by the term ""third party"" b...",R8
4,giuliaclemente26_uq_w050,the-university-of-queensland,giuliaclemente26_uq,Managing Profile and Security Settings,"All right, now. Translate. So Google Authentic...",thinking_aloud,yes,F4,medium,negative,medium,Tester finds the Google Authenticator setup di...,R8


In [14]:
# To check required columns
required_cols = ["window_id", "friction_type", "severity", "sentiment"]

missing_r3 = [col for col in required_cols if col not in r3.columns]
missing_r8 = [col for col in required_cols if col not in r8.columns]

print("Missing columns in R3:", missing_r3)
print("Missing columns in R8:", missing_r8)

if missing_r3 or missing_r8:
    raise ValueError("Required columns are missing. Please check the input files.")

Missing columns in R3: []
Missing columns in R8: []


In [15]:
# Standardise friction_type so both files use the short code format: F1-F7
# Example:
# "F1 Navigation / Findability Issue" -> "F1"
# "F2 Content Clarity Issue" -> "F2"

r3["friction_type"] = r3["friction_type"].astype(str).str.extract(r"^(F\d)")
r8["friction_type"] = r8["friction_type"].astype(str).str.extract(r"^(F\d)")

print("R3 friction_type unique values:", sorted(r3["friction_type"].dropna().unique()))
print("R8 friction_type unique values:", sorted(r8["friction_type"].dropna().unique()))

R3 friction_type unique values: ['F1', 'F2', 'F3', 'F4', 'F5', 'F6', 'F7']
R8 friction_type unique values: ['F1', 'F2', 'F4', 'F5', 'F6', 'F7']


In [17]:
# To check for missing values in key columns
print("Missing values in R3:")
print(r3[required_cols].isna().sum())
print()

print("Missing values in R8:")
print(r8[required_cols].isna().sum())

Missing values in R3:
window_id        0
friction_type    0
severity         0
sentiment        0
dtype: int64

Missing values in R8:
window_id        0
friction_type    0
severity         0
sentiment        0
dtype: int64


In [20]:
# To Keep only needed columns and merge by window_id
r3_small = r3[required_cols].copy()
r8_small = r8[required_cols].copy()

merged = pd.merge(
    r3_small,
    r8_small,
    on="window_id",
    suffixes=("_r3", "_r8")
)

print("Matched windows:", len(merged))
display(merged.head())

Matched windows: 14


,window_id,friction_type_r3,severity_r3,sentiment_r3,friction_type_r8,severity_r8,sentiment_r8
0,ramazankawish_wa_w075,F1,medium,negative,F1,medium,negative
1,gameoverdan_suncorp_w040,F1,medium,negative,F1,low,mixed
2,Sharelinsonny_uq_w026,F2,medium,negative,F2,medium,negative
3,oliviamitchell22_suncorp_w007,F2,medium,mixed,F2,low,mixed
4,giuliaclemente26_uq_w050,F3,medium,negative,F4,medium,negative


In [21]:
# To Check matched window IDs:
print("R3 unique windows:", r3_small["window_id"].nunique())
print("R8 unique windows:", r8_small["window_id"].nunique())
print("Merged unique windows:", merged["window_id"].nunique())

R3 unique windows: 14
R8 unique windows: 14
Merged unique windows: 14


## Compute Cohen's kappa

In [23]:
dimensions = ["friction_type", "severity", "sentiment"]

results = []

for dim in dimensions:
    kappa = cohen_kappa_score(
        merged[f"{dim}_r3"],
        merged[f"{dim}_r8"]
    )
    results.append({
        "dimension": dim,
        "kappa": round(kappa, 4)
    })

results_df = pd.DataFrame(results)
display(results_df)

,dimension,kappa
0,friction_type,0.6606
1,severity,0.3869
2,sentiment,0.6640


In [24]:
for dim in dimensions:
    print(f"\n=== {dim} agreement table ===")
    display(
        pd.crosstab(
            merged[f"{dim}_r3"],
            merged[f"{dim}_r8"],
            rownames=["R3"],
            colnames=["R8"]
        )
    )


=== friction_type agreement table ===


R8,F1,F2,F4,F5,F6,F7
R3,,,,,,
F1,2,0,0,0,0,0
F2,0,2,0,0,0,0
F3,0,0,1,1,0,0
F4,0,0,2,0,0,0
F5,0,0,0,0,1,0
F6,0,0,0,0,1,1
F7,0,0,0,0,0,3



=== severity agreement table ===


R8,high,low,medium,none
R3,,,,
high,1,0,1,0
medium,0,4,4,1
none,0,0,0,3



=== sentiment agreement table ===


R8,mixed,negative,positive
R3,,,
mixed,3,2,0
negative,1,5,0
positive,0,0,3


In [25]:
severity_weighted_kappa = cohen_kappa_score(
    merged["severity_r3"],
    merged["severity_r8"],
    weights="quadratic"
)

print("Weighted kappa for severity:", round(severity_weighted_kappa, 4))

Weighted kappa for severity: 0.6038


## Interpretation

Cohen's kappa measures inter-annotator agreement beyond chance.

General interpretation:
- < 0.00: poor
- 0.00–0.20: slight
- 0.21–0.40: fair
- 0.41–0.60: moderate
- 0.61–0.80: substantial
- 0.81–1.00: almost perfect

Notes:
- `friction_type` is standardised to the short code format (`F1`–`F7`) before agreement is calculated.
- `severity` is ordinal (`none < low < medium < high`), so a weighted kappa is also reported as a supplementary measure.

In [27]:
# To summarise agreement results
friction_kappa = results_df.loc[results_df["dimension"] == "friction_type", "kappa"].values[0]
severity_kappa = results_df.loc[results_df["dimension"] == "severity", "kappa"].values[0]
sentiment_kappa = results_df.loc[results_df["dimension"] == "sentiment", "kappa"].values[0]

print(
    f"Round 1 agreement results: friction_type kappa = {friction_kappa:.4f}, "
    f"severity kappa = {severity_kappa:.4f}, "
    f"sentiment kappa = {sentiment_kappa:.4f}, "
    f"weighted severity kappa = {severity_weighted_kappa:.4f}."
)

Round 1 agreement results: friction_type kappa = 0.6606, severity kappa = 0.3869, sentiment kappa = 0.6640, weighted severity kappa = 0.6038.
